In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import numpy as np
import pickle

MOUNT GOOGLE DRIVE


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Dataset Path

In [ ]:
data_dir = "/content/drive/MyDrive/food_dataset"


Data Augmentation

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2   # 🔥 automatic split
)

train_data = train_datagen.flow_from_directory(
    data_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    data_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 2147 images belonging to 36 classes.
Found 524 images belonging to 36 classes.


Build Model

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

ADD CUSTOM LAYERS

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
output = Dense(len(train_data.class_indices), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

Compile Model

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Train Model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30,
    callbacks=[early_stop]
)

Epoch 1/30
 3/68 ━━━━━━━━━━━━━━━━━━━━ 22s 352ms/step - accuracy: 0.0226 - loss: 3.8693

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


68/68 ━━━━━━━━━━━━━━━━━━━━ 149s 2s/step - accuracy: 0.0554 - loss: 3.5792 - val_accuracy: 0.1031 - val_loss: 3.2513
Epoch 2/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 87s 1s/step - accuracy: 0.1486 - loss: 3.1877 - val_accuracy: 0.4160 - val_loss: 2.8500
Epoch 3/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 89s 1s/step - accuracy: 0.2892 - loss: 2.8525 - val_accuracy: 0.5706 - val_loss: 2.5164
Epoch 4/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 87s 1s/step - accuracy: 0.3996 - loss: 2.5347 - val_accuracy: 0.6183 - val_loss: 2.2227
Epoch 5/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.4928 - loss: 2.2901 - val_accuracy: 0.6622 - val_loss: 1.9816
Epoch 6/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 143s 1s/step - accuracy: 0.5431 - loss: 2.0663 - val_accuracy: 0.7061 - val_loss: 1.7964
Epoch 7/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.5873 - loss: 1.8897 - val_accuracy: 0.7195 - val_loss: 1.5972
Epoch 8/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.6344 - loss: 1.7210 - val_accuracy: 0.7634 - val_loss: 1.4498
E

SAVE MODEL

In [ ]:
model.save("/content/drive/MyDrive/food_model_2.h5")